# 99 — Figure and Table Export Audit

Inventory manuscript and supplement exports, identify competing copies and archive explicitly selected files. Run from the repository root or its `notebooks/` directory. The inventory scans `outputs/`, `notebooks/`, and `playground_notebooks/`. Reports use repository-relative paths and are written to `outputs/export_audit/`.

The manifest identifies the known notebook outputs and records gaps in source coverage. Compare against the revised manuscript assets or enter the confirmed mappings after checking the final PDFs. The original submission establishes the legacy numbering and cannot establish the final placement of revision figures.

In [5]:
from collections import defaultdict
from datetime import datetime
from hashlib import sha256
from pathlib import Path
import csv
import shutil

from IPython.display import display
import pandas as pd

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "notebooks").is_dir() and (directory / "datasets").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the kernel from the repository or its notebooks folder. "
        "The repository must contain sibling notebooks/ and datasets/ folders."
    )

EXTRA_EXPORT_ROOTS = []  # Optional additional export folders within the repository.
AUDIT_DIR = PROJECT_ROOT / "outputs" / "export_audit"
ARCHIVE_DIR = PROJECT_ROOT / "outputs" / "archived_exports"

# Optional: map a manuscript ID to an authoritative source asset used in the final paper.
# Use a repository-relative path. For a four-panel export, keep the panel IDs separate.
REFERENCE_ASSETS = {
    # "Figure 6a": "manuscript/figures/fig_06a_native_daily_r.png",
}
# For PDF-only checks, record the exact path whose content was visually verified.
VERIFIED_IN_PDF = {
    # "Figure 6a": "outputs/revised_plots/fig_06a_native_daily_r.png",
}

## Reset the repository's main outputs folder

After running the setup above, set `RESET_OUTPUTS = True` and run only the reset cell below. It moves the entire repository-level `outputs/` folder into a timestamped `output_backups/` folder and recreates an empty `outputs/`. Set the flag back to `False` afterward.

Use `INCLUDE_LEGACY_OUTPUTS = True` only if you also want to reset `notebooks/outputs/` and `playground_notebooks/outputs/`. The inventory cells below create `outputs/export_audit/`. Run them after regenerating figures and tables if you want the main folder to stay empty until then.

In [6]:
RESET_OUTPUTS = False
INCLUDE_LEGACY_OUTPUTS = False

main_output = PROJECT_ROOT / "outputs"
reset_candidates = [main_output]
if INCLUDE_LEGACY_OUTPUTS:
    reset_candidates.extend(
        PROJECT_ROOT / directory / "outputs"
        for directory in ("notebooks", "playground_notebooks")
    )

reset_plan = [
    folder for folder in reset_candidates
    if folder.is_dir() and (folder != main_output or any(folder.iterdir()))
]
display(pd.DataFrame({
    "output_folder": [
        folder.relative_to(PROJECT_ROOT).as_posix() for folder in reset_plan
    ],
}))

if RESET_OUTPUTS:
    for source in reset_candidates:
        if source.is_symlink() or (source.exists() and not source.is_dir()):
            raise ValueError(f"Expected a real output folder at {source.relative_to(PROJECT_ROOT)}.")
        if not source.parent.resolve().is_relative_to(PROJECT_ROOT):
            raise ValueError(f"Output folder must be inside the repository: {source.relative_to(PROJECT_ROOT)}.")

    if reset_plan:
        backup_root = PROJECT_ROOT / "output_backups"
        if backup_root.is_symlink():
            raise ValueError("Expected a real output_backups/ directory.")
        reset_backup = backup_root / datetime.now().strftime("%Y%m%dT%H%M%S%f")
        reset_backup.mkdir(parents=True, exist_ok=False)
        moved = []
        try:
            for source in reset_plan:
                destination = reset_backup / source.relative_to(PROJECT_ROOT)
                destination.parent.mkdir(parents=True, exist_ok=True)
                source.rename(destination)
                moved.append((source, destination))
            main_output.mkdir(exist_ok=True)
        except OSError:
            for source, destination in reversed(moved):
                destination.rename(source)
            raise
        print(f"Previous outputs moved to {reset_backup.relative_to(PROJECT_ROOT)}/")

    else:
        main_output.mkdir(exist_ok=True)
    print("Repository-level outputs/ is empty and ready for new exports.")


,output_folder
0,outputs


## 1. Manuscript and supplement manifest

Each row maps a manuscript item or revision diagnostic to expected export stems. The original submission supplied IDs for Figures 1–8 and S1–S8. The updated heatmaps and time series use the manuscript-facing stems from the revision notebooks. For revision-only outputs without confirmed final numbering, the manifest uses a descriptive label. Update these labels after checking the revised documents.

In [7]:
MANIFEST = [
    ("Figure 1", "Study region and analysis units", "", (), "Source generator unavailable"),
    ("Figure 2", "Reliability-qualification workflow diagram", "", (), "Source generator unavailable"),
    ("Figure 3a", "VNP46A2 panel, 1 Nov 2013", "10", ("fig_03_20131101",), "Revised candidate"),
    ("Figure 3b", "VNP46A2 panel, 5 Nov 2013", "10", ("fig_03_20131105",), "Revised candidate"),
    ("Figure 3c", "VNP46A2 panel, 10 Nov 2013", "10", ("fig_03_20131110",), "Revised candidate"),
    ("Figure 3d", "VNP46A2 panel, 13 Nov 2013", "10", ("fig_03_20131113",), "Confirm in revised layout"),
    ("Figure 4", "POI temporal and spatial CV", "11", ("fig_04_poi_cv_by_kernel_and_settlement_2013",), "Revised source"),
    ("Figure 5", "Global temporal completeness", "", (), "Source generator unavailable"),
    ("Figure 6a", "Native-daily Pearson r", "13", ("fig_06a_native_daily_r", "section1_native_daily_pearson_r"), "Revised source"),
    ("Figure 6b", "Native-daily retained N%", "13", ("fig_06b_native_daily_N", "section1_native_daily_direct_N"), "Revised source"),
    ("Figure 6c", "Native-daily composite score", "13", ("fig_06c_native_daily_score", "section1_native_daily_composite_score"), "Revised source"),
    ("Figure 7", "Samar–Leyte NTL–load alignment and Haiyan zoom", "13", ("fig_07_samar_leyte",), "Revised source"),
    ("Figure 8", "Regional NTL–load alignment and Rai zoom", "14", ("fig_08_selected_regions_rai",), "Revised source"),
    ("Regional component trade-off", "Daily alignment versus coverage by sub-grid", "14", ("fig_native_daily_alignment_coverage",), "Confirm final ID"),
    ("Figure 10", "Window sensitivity at fixed native-daily configurations", "15", ("fig_10_fixed_configuration_window_sensitivity",), "Source figure label"),
    ("Table 1", "GHSL classes and analysis groups", "12", ("table_01_ghsl_smod_classes_and_analysis_groups",), "Revised source"),
    ("Table 2", "POI kernel observability", "11", ("table_02_poi_kernel_observability_2013",), "Revised source"),
    ("Table 3", "Settlement-scale observability", "12", ("table_03_settlement_observability_diagnostics_2013",), "Revised source"),
    ("Regional ranking table", "Top-five native-daily configurations per sub-grid", "14", ("section2_top5_daily_configurations_by_subgrid_5pct_tau",), "Confirm Table 4/Supplement placement"),
    ("Figure S1", "Regional radiance and spatial completeness", "10", ("fig_s1_regional_dnb_gapfilled_completeness_2013",), "Original ID"),
    ("Figure S2", "Tacloban 1×1 observability", "11", ("fig_sx_tacloban_dnb_gapfilled_by_kernel_2013",), "Related export spans all three supports"),
    ("Figure S3", "Regional long-term spatial completeness", "10", ("fig_s3_spatial_completeness_2012_2024",), "Original ID"),
    ("Figure S4", "Pre–post Haiyan radiance differences", "11", ("fig_s4_prepost_haiyan_delta_ntl_by_kernel_2013",), "Original ID"),
    ("Figure S5", "POI direct and Gap-Filled time series", "11", ("fig_s5_representative_poi_dnb_gapfilled_by_kernel_2013",), "Original ID"),
    ("Figure S6", "Representative thematic mask maps", "", (), "Source generator unavailable"),
    ("Figure S7", "NTL series across SC thresholds", "13", ("fig_sx_interactive_ntl_reliability_filter_samar_leyte",), "Related interactive export"),
    ("Figure S8a", "Raw NTL–NGCP alignment at τ = 10%", "13", ("fig_s8a_raw_ntl_ngcp_tau10",), "Original runnable workflow"),
    ("Figure S8b", "Raw NTL–NGCP alignment at τ = 50%", "13", ("fig_s8b_raw_ntl_ngcp_tau50",), "Original runnable workflow"),
    ("Figure S8c", "Raw NTL–NGCP alignment at τ = 90%", "13", ("fig_s8c_raw_ntl_ngcp_tau90",), "Original runnable workflow"),
    ("Table S2", "Full POI observability", "11", ("table_s2_full_poi_observability_metrics_2013",), "Revised source"),
    ("Table S3", "Full settlement observability", "12", ("table_s3_full_settlement_observability_metrics_2013",), "Revised source"),
    ("Regional heatmaps", "Regional native-daily component and score surfaces", "14", ("fig_08_regional_daily_heatmaps",), "Confirm final ID"),
    ("Temporal checkpoints", "Mask–threshold checkpoint surfaces for r", "15", ("section1_checkpoint_heatmaps_r",), "Confirm final ID"),
    ("Temporal coverage checkpoints", "Mask–threshold checkpoint surfaces for N%", "15", ("section1_checkpoint_heatmaps_N_pct",), "Confirm final ID"),
    ("Event window comparison", "Haiyan/Rai window sensitivity, main layout", "15", ("section1_event_alignment_main_W1_W4_W30",), "Confirm final ID"),
    ("Event window comparison, expanded", "Haiyan/Rai window sensitivity, supplementary layout", "15", ("section1_event_alignment_supp_W1_W4_W7_W14_W30",), "Confirm final ID"),
    ("Score robustness", "Regional comparison of score formulations", "15", ("section2_composite_score_Nr_vs_Nr2",), "Confirm final ID"),
    ("Score robustness table", "Selected regional settings under each formula", "15", ("section2_composite_score_Nr_vs_Nr2_selections",), "Confirm final ID"),
]

## 2. Inventory and verify

Files with the same stem and extension but different paths are competing exports. Grouping by SHA-256 identifies byte-identical copies within each file format. A visual match from a PDF can be recorded separately from a byte match against the original asset. Files that have no manifest match stay visible for review.

In [8]:
def resolve_path(path):
    path = Path(path).expanduser()
    return (PROJECT_ROOT / path).resolve()


def checksum(path):
    with path.open("rb") as stream:
        return sha256_file(stream)


def sha256_file(stream):
    digest = sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
    return digest.hexdigest()


def stem_and_format(path):
    if path.name.lower().endswith(".csv.gz"):
        return path.name[:-7], ".csv.gz"
    return path.stem, path.suffix.lower()


scan_roots = [
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "playground_notebooks",
]
scan_roots.extend(resolve_path(path) for path in EXTRA_EXPORT_ROOTS)
scan_roots = list(dict.fromkeys(scan_roots))
allowed_formats = {".png", ".svg", ".pdf", ".html", ".jpg", ".jpeg", ".tif", ".tiff",
                   ".csv", ".csv.gz", ".tsv", ".xlsx", ".parquet"}
files = {}
for root in scan_roots:
    if not root.is_relative_to(PROJECT_ROOT) or not root.is_dir():
        continue
    for path in root.rglob("*"):
        stem, suffix = stem_and_format(path)
        if suffix not in allowed_formats or path.is_symlink() or not path.is_file():
            continue
        path = path.resolve()
        if (
            not path.is_relative_to(PROJECT_ROOT)
            or not path.is_relative_to(root)
            or any(path.is_relative_to(folder) for folder in (
                AUDIT_DIR, ARCHIVE_DIR, PROJECT_ROOT / "output_backups"
            ))
        ):
            continue
        files[path] = (stem.casefold(), suffix)

records = []
by_stem = defaultdict(list)
for path, (stem, suffix) in sorted(files.items()):
    row = {"path": path.relative_to(PROJECT_ROOT).as_posix(), "stem": stem, "format": suffix,
           "bytes": path.stat().st_size, "sha256": checksum(path)}
    by_stem[stem].append(row)
    records.append(row)

inventory = pd.DataFrame(records, columns=["path", "stem", "format", "bytes", "sha256"])
inventory["same_stem_format"] = inventory.duplicated(["stem", "format"], keep=False)
inventory["byte_duplicate"] = inventory.duplicated(["format", "sha256"], keep=False)

audit_rows = []
protected = set()
for manuscript_id, description, notebook, stems, mapping in MANIFEST:
    candidate_paths = {
        row["path"]: row for stem in stems for row in by_stem[stem.casefold()]
    }
    reference = resolve_path(REFERENCE_ASSETS[manuscript_id]) if manuscript_id in REFERENCE_ASSETS else None
    verified = resolve_path(VERIFIED_IN_PDF[manuscript_id]) if manuscript_id in VERIFIED_IN_PDF else None
    protected.update(path for path in (reference, verified) if path is not None)
    for path in (reference, verified):
        if path in files:
            stem = files[path][0]
            relative_path = path.relative_to(PROJECT_ROOT).as_posix()
            row = next(row for row in by_stem[stem] if row["path"] == relative_path)
            candidate_paths[relative_path] = row
    reference_digest = checksum(reference) if reference is not None and reference.is_file() else None
    candidates = list(candidate_paths.values())
    if not candidates:
        status = "No mapped generator" if not stems else "Export missing"
        audit_rows.append((manuscript_id, description, notebook, mapping, "", status))
        continue
    for row in candidates:
        path = PROJECT_ROOT / row["path"]
        protected.add(path)
        if reference_digest and row["sha256"] == reference_digest:
            status = "Byte match"
        elif verified == path:
            status = "Verified in PDF"
        elif reference_digest:
            status = "Different bytes; compare format or layout"
        elif reference is not None:
            status = "Reference asset missing"
        else:
            status = "Candidate; verify against final paper"
        audit_rows.append((manuscript_id, description, notebook, mapping, row["path"], status))

audit = pd.DataFrame(audit_rows, columns=["paper_id", "description", "notebook", "mapping", "path", "status"])
inventory["mapped_or_protected"] = inventory["path"].isin({path.relative_to(PROJECT_ROOT).as_posix() for path in protected if path.is_relative_to(PROJECT_ROOT)})

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
inventory.to_csv(AUDIT_DIR / "export_inventory.csv", index=False)
audit.to_csv(AUDIT_DIR / "paper_output_audit.csv", index=False)
display(audit)
display(inventory.loc[inventory["same_stem_format"] | inventory["byte_duplicate"]])

,paper_id,description,notebook,mapping,path,status
0,Figure 1,Study region and analysis units,,Source generator unavailable,,No mapped generator
1,Figure 2,Reliability-qualification workflow diagram,,Source generator unavailable,,No mapped generator
2,Figure 3a,"VNP46A2 panel, 1 Nov 2013",10,Revised candidate,outputs/figures/main/fig_03_20131101.html,Candidate; verify against final paper
3,Figure 3a,"VNP46A2 panel, 1 Nov 2013",10,Revised candidate,outputs/figures/main/fig_03_20131101.jpeg,Candidate; verify against final paper
4,Figure 3a,"VNP46A2 panel, 1 Nov 2013",10,Revised candidate,outputs/figures/main/fig_03_20131101.png,Candidate; verify against final paper
...,...,...,...,...,...,...
115,Score robustness,Regional comparison of score formulations,15,Confirm final ID,outputs/figures/supplementary/section2_composi...,Candidate; verify against final paper
116,Score robustness,Regional comparison of score formulations,15,Confirm final ID,outputs/figures/supplementary/section2_composi...,Candidate; verify against final paper
117,Score robustness,Regional comparison of score formulations,15,Confirm final ID,outputs/figures/supplementary/section2_composi...,Candidate; verify against final paper
118,Score robustness,Regional comparison of score formulations,15,Confirm final ID,outputs/figures/supplementary/section2_composi...,Candidate; verify against final paper


,path,stem,format,bytes,sha256,same_stem_format,byte_duplicate,mapped_or_protected
0,outputs/figures/diagnostics/poi_alang_alang_20...,poi_alang_alang_2013,.html,4446793,3c77c41d07eba7979e3e9fa4ad1422871cdd79cf66086c...,True,False,False
2,outputs/figures/diagnostics/poi_alang_alang_20...,poi_alang_alang_2013,.png,228968,91cfd8864f28316b5227fed28f216e2e17be20ab84d4f6...,True,False,False
4,outputs/figures/diagnostics/poi_baybay_city_20...,poi_baybay_city_2013,.html,4444325,eb7c0854907da3e621586a7319e8fed728ebbedf860c46...,True,False,False
6,outputs/figures/diagnostics/poi_baybay_city_20...,poi_baybay_city_2013,.png,237899,ac8394d8e31eda5952a9fce3e5a39239d6ebdea21308ac...,True,False,False
8,outputs/figures/diagnostics/poi_central_leyte_...,poi_central_leyte_forest_2013,.html,4446741,4a69d71e3a5ef316fcb638ebef84c0dd5f27e72ed0bb4d...,True,False,False
...,...,...,...,...,...,...,...,...
423,playground_notebooks/outputs/notebook06_rebuil...,poi_samar_forest_reserve_2013,.png,171030,8368a29e9997a4bf907fb123d7ae9f69df7ebed31ae1d3...,True,False,False
424,playground_notebooks/outputs/notebook06_rebuil...,poi_southern_leyte_gulf_2013,.html,128024,61a24f82a8ce2ddd0c6c466d49753465a258b645ab83cf...,True,False,False
425,playground_notebooks/outputs/notebook06_rebuil...,poi_southern_leyte_gulf_2013,.png,162551,bff76ddc4effd21d5f3d223fc39a8db492c9698c3d96fd...,True,False,False
426,playground_notebooks/outputs/notebook06_rebuil...,poi_tacloban_city_2013,.html,119396,69dbc55b87e37bb255c1bdd9e1ffb694d227166387aa6a...,True,False,False


## 3. Review and archive selected exports

Use the inventory to identify obsolete paths. Add exact repository-relative paths to `APPROVED_ARCHIVE` and review the plan with `APPLY_ARCHIVE = False`. For any manifest candidate you want to archive, first narrow its manifest mapping or record the authoritative asset and specify a confirmed path in `VERIFIED_IN_PDF`. Once the plan is reviewed, set `APPLY_ARCHIVE = True` to move the listed files to a dated folder. The move log records the original path and checksum.

In [9]:
APPROVED_ARCHIVE = []
APPLY_ARCHIVE = False

approved = list(dict.fromkeys(resolve_path(path) for path in APPROVED_ARCHIVE))
confirmed_assets = {
    resolve_path(path) for path in [*REFERENCE_ASSETS.values(), *VERIFIED_IN_PDF.values()]
}
verified_ids = set(REFERENCE_ASSETS) | set(VERIFIED_IN_PDF)
unresolved_candidates = {
    resolve_path(path) for path in audit.loc[
        ~audit["paper_id"].isin(verified_ids) | audit["status"].isin(["Byte match", "Verified in PDF", "Reference asset missing"]),
        "path",
    ] if path
}
protected_for_archive = confirmed_assets | unresolved_candidates
known_files = inventory.set_index("path")["sha256"].to_dict()
batch_dir = ARCHIVE_DIR / datetime.now().strftime("%Y%m%dT%H%M%S%f")

plan = []
for path in approved:
    if not path.is_relative_to(PROJECT_ROOT):
        raise ValueError("Archive candidates must be inside the repository.")
    relative = path.relative_to(PROJECT_ROOT).as_posix()
    if relative not in known_files or path in protected_for_archive or not path.is_file():
        raise ValueError(f"File is outside the current inventory, protected, or missing: {relative}")
    if checksum(path) != known_files[relative]:
        raise ValueError(f"File changed after inventory. Refresh the audit first: {relative}")
    destination = (batch_dir / relative).relative_to(PROJECT_ROOT).as_posix()
    plan.append({"source": relative, "destination": destination, "sha256": known_files[relative]})

archive_plan = pd.DataFrame(plan, columns=["source", "destination", "sha256"])
display(archive_plan)

if APPLY_ARCHIVE and plan:
    batch_dir.mkdir(parents=True, exist_ok=False)
    with (batch_dir / "move_log.csv").open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["source", "destination", "sha256"])
        writer.writeheader()
        for row in plan:
            destination = PROJECT_ROOT / row["destination"]
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(PROJECT_ROOT / row["source"], destination)
            writer.writerow(row)
            stream.flush()

,source,destination,sha256


In [10]:
from zipfile import ZipFile, ZIP_DEFLATED

export_root = PROJECT_ROOT / "outputs"
zip_path = export_root / "figures_and_tables.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for folder in ("figures", "tables"):
        for path in sorted((export_root / folder).rglob("*")):
            if path.is_file() and (
                folder == "tables" or path.suffix.lower() in {".jpeg", ".jpg"}
            ):
                archive.write(path, arcname=path.relative_to(export_root).as_posix())

print(f"Created: {zip_path.relative_to(PROJECT_ROOT)}")

Created: outputs/figures_and_tables.zip
